# Quantum Optimization for Distributed Order Management (DOM)

## Notebook 03 – Classical Baseline Model

### Objective

The purpose of this notebook is to establish a classical baseline for the Distributed Order Management (DOM) problem.

This baseline represents the existing fulfillment strategy, where customer orders remain assigned to their default Distribution Centers whenever possible.

The baseline provides a reference against which advanced optimization methods, including quantum and hybrid approaches, will be evaluated.

In [1]:
import pandas as pd
import numpy as np

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


# Load Prepared Datasets

The cleaned datasets generated in the previous notebooks are loaded for baseline evaluation.

These datasets contain customer orders, warehouse capacities, and shipping information required to evaluate the existing fulfillment strategy.

# Load Prepared Datasets

This notebook uses the cleaned and supporting datasets prepared in the previous notebooks.

The datasets represent different operational aspects of the Distributed Order Management (DOM) problem.

| Dataset | Purpose |
|----------|----------|
| orders_clean.csv | Cleaned customer order data |
| input_capacity_planning.csv | Distribution Center capacity planning |
| input_dock_capacity.csv | Dock capacity constraints |
| input_shipping_cost_data.csv | Transportation cost information |
| input_throughput_capacity.csv | Warehouse processing capacity |

These datasets are loaded to evaluate the baseline order assignment strategy before applying optimization algorithms.

In [2]:
orders = pd.read_csv("../data/orders_clean.csv")

capacity = pd.read_csv("../data/input data/input_capacity_planning.csv")

dock = pd.read_csv("../data/input data/input_dock_capacity.csv")

shipping = pd.read_csv("../data/input data/input_shipping_cost_data.csv")

throughput = pd.read_csv("../data/input data/input_throughput_capacity.csv")

## Verify Dataset Dimensions

Before building the baseline model, we verify that all required datasets have been loaded successfully.

The dataset dimensions provide confidence that the optimization workflow is using complete input data.

In [3]:
print("Orders:", orders.shape)
print("Capacity:", capacity.shape)
print("Dock:", dock.shape)
print("Shipping:", shipping.shape)
print("Throughput:", throughput.shape)

Orders: (25193, 36)
Capacity: (377504, 23)
Dock: (480, 13)
Shipping: (12922, 7)
Throughput: (530, 7)


## Preview Order Data

The first few records are displayed to verify that the cleaned order dataset has been loaded correctly.

This dataset forms the basis of the baseline assignment model.

In [4]:
orders.head()

,Group_Flag,Plant,MaterialNumber,transportationplanningdate,IsTopCust,OpeningStock,RequestedDeliveryDate,DeliveryNoteFlag,IsInvAvail,LoadNumber,...,IsMultipleRDD,Measure,FillRateThreshold,Penaltyforpotentialcuts,MaximumPenalty,FixedPenalty,FixedPenaltyPerSKU,MinimumPenalty,OnTimePercentage,OnTimeFixed
0,5484913123,5083,12260382,6/26/24,N,33814.0,6/27/24,N,Y,U600105191,...,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5484913123,5083,9516458,6/26/24,N,6546.0,6/27/24,N,Y,U600105191,...,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5484913123,5083,12408924,6/26/24,N,4151.0,6/27/24,N,Y,U600105191,...,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5484913123,5083,9517630,6/26/24,N,17667.0,6/27/24,N,Y,U600105191,...,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5484913123,5083,12587091,6/26/24,N,89341.0,6/27/24,N,Y,U600105191,...,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Observation

The cleaned order dataset contains the information required to evaluate the default fulfillment strategy.

## Preview Capacity Dataset

Warehouse capacity information is inspected before evaluating the baseline.

Capacity constraints are important because Distribution Centers have limited operational resources.

In [5]:
capacity.head()

,LocationID,MaterialID,DATE,OpeningStock,SalesActualCustomerOrders,Total_Unreserved_Qty,ClosingStock,TotalSupply,BatchAvailability,OutgoingDispatchPlan,...,EndOfShelfLife,TotalDemand,ReferenceDemand,OutgoingLoadPlan,Total_Reserved_Qty,Available_inventory,LeftOver_Qty,CumIncSTO,report_date,SalesOrderDemand
0,5081,12321261,2024-06-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-06-21,NaN
1,5081,12321261,2024-06-21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-06-21,NaN
2,5081,12321261,2024-06-22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-06-21,NaN
3,5081,12321261,2024-06-23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-06-21,NaN
4,5081,12321261,2024-06-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-06-21,NaN


## Preview Shipping Dataset

Shipping information is reviewed because transportation cost is one of the primary optimization objectives.

The baseline model will later be compared with optimized shipping decisions.

In [6]:
shipping.head()

,TargetZip,OrigZip3,CostPerLoadAmbientOutboundMoves,FuelSurchargeAmbient,Distance,Shipping_Cost,Plant
0,786,302,1538,454,966,1992,5082
1,786,302,1538,454,966,1992,5410
2,320,302,916,143,305,1059,5082
3,320,302,916,143,305,1059,5410
4,331,302,1753,319,678,2072,5082


## Distribution Center Summary

This section summarizes the available Distribution Centers participating in order fulfillment.

Understanding the warehouse network is essential before evaluating baseline assignments.

In [7]:
print("Unique Warehouses:", orders["Plant"].nunique())
print(orders["Plant"].unique())

Unique Warehouses: 8
[5083 5620 5385 5410 5420 5490 5641 5773]


## Orders per Distribution Center

The number of customer orders assigned to each Distribution Center is calculated.

This provides an overview of workload distribution under the default assignment strategy.

In [8]:
orders["Plant"].value_counts()

Plant
5385    6944
5420    5280
5620    3790
5410    3447
5490    2264
5083    2236
5641     995
5773     237
Name: count, dtype: int64

## Total Order Quantity by Distribution Center

This analysis measures the total demand handled by each Distribution Center.

These values provide insight into warehouse utilization.

In [9]:
orders.groupby("Plant")["OrderedQty_converted"].sum()

Plant
5083    140303
5385    551682
5410    280171
5420    593529
5490    236411
5620    402089
5641    128612
5773     12816
Name: OrderedQty_converted, dtype: int64

## Average Order Quantity

The average quantity per order is calculated for each Distribution Center.

Average demand helps compare operational characteristics across warehouses.

In [10]:
orders.groupby("Plant")["OrderedQty_converted"].mean()

Plant
5083     62.747317
5385     79.447293
5410     81.279663
5420    112.410795
5490    104.421820
5620    106.092084
5641    129.258291
5773     54.075949
Name: OrderedQty_converted, dtype: float64

## Revenue by Distribution Center

The total revenue associated with each Distribution Center is calculated.

This information helps understand the business value supported by each warehouse.

In [11]:
orders.groupby("Plant")["Order_SKU_Revenue"].sum()

Plant
5083     8622132
5385    18451031
5410     9517592
5420    20741183
5490     7256570
5620    18457010
5641     4553608
5773      892735
Name: Order_SKU_Revenue, dtype: int64

## Revenue Ranking

Distribution Centers are ranked according to the revenue generated from customer orders.

This ranking identifies high-value fulfillment locations.

In [12]:
orders.groupby("Plant")["Order_SKU_Revenue"]\
      .sum()\
      .sort_values(ascending=False)\
      .head(10)

Plant
5420    20741183
5620    18457010
5385    18451031
5410     9517592
5083     8622132
5490     7256570
5641     4553608
5773      892735
Name: Order_SKU_Revenue, dtype: int64

# Build the Classical Baseline

The baseline model assumes that customer orders remain assigned to their default Distribution Centers.

No optimization or reassignment is performed.

This represents the reference solution against which improved optimization methods will be compared.

In [13]:
baseline = orders.copy()

baseline["Assigned_DC"] = baseline["Plant"]

baseline.head()

,Group_Flag,Plant,MaterialNumber,transportationplanningdate,IsTopCust,OpeningStock,RequestedDeliveryDate,DeliveryNoteFlag,IsInvAvail,LoadNumber,...,Measure,FillRateThreshold,Penaltyforpotentialcuts,MaximumPenalty,FixedPenalty,FixedPenaltyPerSKU,MinimumPenalty,OnTimePercentage,OnTimeFixed,Assigned_DC
0,5484913123,5083,12260382,6/26/24,N,33814.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5083
1,5484913123,5083,9516458,6/26/24,N,6546.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5083
2,5484913123,5083,12408924,6/26/24,N,4151.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5083
3,5484913123,5083,9517630,6/26/24,N,17667.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5083
4,5484913123,5083,12587091,6/26/24,N,89341.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5083


## Baseline Performance Metrics

Several key performance indicators (KPIs) are calculated to evaluate the baseline strategy.

These metrics include:

- Fill Rate
- Shipping Cost
- Revenue
- Assignment Quality

The same metrics will later be used to compare optimization approaches.

In [14]:
fill_rate = (
    baseline["IsInvAvail"]
    .eq("Y")
    .mean()
    * 100
)

print(f"Baseline Fill Rate: {fill_rate:.2f}%")

Baseline Fill Rate: 93.68%


### Observation

The calculated KPIs establish the performance of the default fulfillment strategy.

Future optimization methods should improve one or more of these metrics while satisfying operational constraints.

## Save Baseline Results

The baseline assignment results are saved for use in subsequent notebooks.

These results provide a consistent benchmark for evaluating optimization algorithms.

In [15]:
baseline.to_csv(
    "../data/baseline_assignment.csv",
    index=False
)

print("Baseline assignment saved successfully!")

Baseline assignment saved successfully!


# Conclusion

In this notebook, a transparent classical baseline for the Distributed Order Management problem was established.

The baseline retains the default order assignments and evaluates their performance using key business metrics.

This baseline serves as the benchmark against which heuristic, mathematical optimization, and quantum optimization methods will be compared in the following notebooks.